# Example 02: Visium Spatial Neighbor Graph（增强版）

本 notebook 从 Visium 空间坐标构建 k-NN 邻域图并可视化。

**Phase 0 增强图型：**
- 按组织区域着色的空间图
- 边距离分布直方图 + KDE

In [ ]:
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path.cwd().parent))

from src.data.loaders import load_visium_spatial
from src.data.validators import validate_visium_positions
from src.examples.config import VISIUM_DIR, SPATIAL_K
from src.examples.spatial_graph import (
    build_knn_edges,
    filter_in_tissue,
    graph_stats,
    plot_spatial_graph_colored,
    plot_edge_distance_distribution,
)
from src.utils.plot_style import apply_style
apply_style()

In [ ]:
# 加载与校验
positions, scalefactors = load_visium_spatial(VISIUM_DIR)
print(f"总 spot 数: {len(positions)}")
print(f"缩放因子: {scalefactors}")
print(f"校验: {validate_visium_positions(positions) or 'OK'}")

In [ ]:
# 过滤 in_tissue
tissue = filter_in_tissue(positions)
print(f"in_tissue spot: {len(tissue)}")
coords = tissue[["pxl_col_in_fullres", "pxl_row_in_fullres"]].values

In [ ]:
# 构建 k-NN 图
edges = build_knn_edges(coords, k=SPATIAL_K)
stats = graph_stats(edges, n_nodes=len(tissue))
print(f"图统计: {stats}")
edges.head()

In [ ]:
# 可视化空间图
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))

# 采样边
edge_sample = edges.sample(min(5000, len(edges)), random_state=42)
for _, row in edge_sample.iterrows():
    s, t = int(row["source"]), int(row["target"])
    ax.plot(
        [coords[s, 0], coords[t, 0]],
        [coords[s, 1], coords[t, 1]],
        color="lightgray", linewidth=0.3, zorder=1,
    )
ax.scatter(coords[:, 0], coords[:, 1], s=3, c="steelblue", zorder=2)
ax.set_aspect("equal")
ax.set_title(f"Visium Spatial Graph (k={SPATIAL_K})")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

## Phase 0 增强可视化

In [ ]:
# 按组织区域着色
if "array_row" in tissue.columns:
    row_vals = tissue["array_row"].values
    bins = np.linspace(row_vals.min(), row_vals.max() + 1, 6)
    region_labels = np.array([f"Region {r}" for r in np.digitize(row_vals, bins)])
    fig = plot_spatial_graph_colored(coords, edges, region_labels, label_name="Tissue Region")
    plt.show()

In [ ]:
# 边距离分布
fig = plot_edge_distance_distribution(edges)
plt.show()